# 01 — Démonstration du pipeline

Ce notebook présente rapidement la V1 du projet **AI Job Application Agent**.

L'objectif n'est pas de faire une analyse très longue, mais de montrer le fonctionnement complet du prototype :

- chargement des offres d'emploi ;
- scoring selon un profil cible ;
- classement des offres ;
- génération de recommandations ;
- génération de messages de candidature ;
- export des résultats.

Cette V1 est volontairement simple : elle repose sur des règles et des mots-clés. Elle sert de base propre avant d'ajouter des briques plus avancées comme la recherche automatique d'offres, la similarité sémantique ou l'utilisation d'un LLM.

## 1. Initialisation

On commence par importer les librairies utiles et par définir les chemins du projet.

Dans le projet, les scripts sont dans `src/`, les données brutes dans `data/raw/`, les données traitées dans `data/processed/`, et les résultats dans `results/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Chemin vers la racine du projet depuis le dossier notebooks/
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

# On ajoute src/ au path Python pour pouvoir importer nos fonctions.
sys.path.append(str(SRC_DIR))

PROJECT_ROOT

## 2. Chargement des offres

Pour la V1, les offres sont encore entrées manuellement dans un fichier CSV.

Dans un vrai usage, cette étape pourra être remplacée plus tard par une recherche automatique via API, scraping ou import depuis LinkedIn/Welcome to the Jungle/Indeed.

In [ ]:
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "job_offers_sample.csv"

job_offers = pd.read_csv(RAW_PATH)
job_offers.head()

Chaque ligne correspond à une offre d'emploi. La colonne la plus importante pour le scoring est `description`, car c'est dans ce texte que l'on cherche les signaux utiles : Python, machine learning, LLM, NLP, santé, finance, reporting, etc.

In [ ]:
job_offers.info()

## 3. Scoring des offres

Le scoring est défini dans le fichier `src/scoring.py`.

L'idée est de séparer le profil cible en plusieurs catégories :

- data science / machine learning ;
- IA générative, LLM, NLP ;
- data engineering et mise en production ;
- santé, biomédical ou recherche ;
- finance et risque ;
- reporting et dashboarding.

Chaque catégorie contient des mots-clés et un poids. Quand un mot-clé apparaît dans une offre, l'offre gagne des points dans la catégorie correspondante.

Le médical est traité comme un bonus, pas comme une contrainte absolue. L'objectif principal reste de repérer des opportunités réalistes en Data Science, ML ou IA.

In [ ]:
from scoring import score_offer

# Exemple sur la première offre
first_description = job_offers.loc[0, "description"]
score_offer(first_description)

La fonction renvoie un dictionnaire avec plusieurs informations :

- le score total ;
- le niveau de recommandation ;
- le type de match ;
- l'action suivante conseillée ;
- une explication lisible ;
- les scores par catégorie ;
- les mots-clés trouvés.

## 4. Application du scoring à toutes les offres

On applique maintenant la fonction de scoring à chaque offre du fichier CSV.

In [ ]:
scoring_results = job_offers["description"].apply(score_offer)
scoring_df = pd.DataFrame(scoring_results.tolist())

jobs_scored = pd.concat([job_offers, scoring_df], axis=1)
jobs_scored = jobs_scored.sort_values("score", ascending=False)

jobs_scored[[
    "title",
    "company",
    "location",
    "score",
    "recommendation",
    "fit_type",
    "next_action"
]]

## 5. Lecture rapide du classement

Le tableau permet de répondre rapidement à la question : **quelle offre mérite mon attention en premier ?**

Une offre avec un score élevé n'est pas seulement une offre qui contient beaucoup de mots-clés. Le score permet aussi de voir quel type de correspondance existe avec le profil : plutôt LLM/NLP, plutôt data science classique, plutôt santé, plutôt finance, ou plutôt reporting.

In [ ]:
for _, row in jobs_scored.iterrows():
    print(f"{row['title']} — {row['company']}")
    print(f"Score : {row['score']} | Recommandation : {row['recommendation']}")
    print(f"Action : {row['next_action']}")
    print(row["explanation"])
    print("-" * 80)

## 6. Génération des messages de candidature

Le fichier `src/application_message.py` contient une fonction qui génère un premier message de candidature à partir des informations scorées.

Pour l'instant, ce n'est pas encore un vrai LLM. C'est un template intelligent qui adapte une phrase selon le type d'offre : LLM/NLP, santé/recherche, finance/risque, engineering, etc.

Cette étape permet déjà de gagner du temps en produisant une première base de message à relire et modifier avant envoi.

In [ ]:
from application_message import generate_application_message

# On génère un message pour la meilleure offre du classement.
best_offer = jobs_scored.iloc[0]
message = generate_application_message(best_offer)
print(message)

On peut aussi générer un message pour chaque offre suffisamment intéressante.

Dans le script, on garde par exemple les offres avec un score supérieur ou égal à 12.

In [ ]:
selected_jobs = jobs_scored[jobs_scored["score"] >= 12].copy()
selected_jobs["application_message"] = selected_jobs.apply(generate_application_message, axis=1)

selected_jobs[["title", "company", "score", "recommendation", "next_action", "application_message"]].head()

## 7. Export des résultats

Le projet exporte deux types de fichiers :

- un CSV pour garder les données structurées ;
- un Markdown pour relire facilement les messages générés.

Le Markdown est plus agréable pour consulter les résultats dans VS Code ou sur GitHub.

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

scored_output_path = PROCESSED_DIR / "scored_job_offers_from_notebook.csv"
messages_output_path = RESULTS_DIR / "application_messages_from_notebook.csv"

jobs_scored.to_csv(scored_output_path, index=False)
selected_jobs.to_csv(messages_output_path, index=False)

print(f"Offres scorées sauvegardées ici : {scored_output_path}")
print(f"Messages sauvegardés ici : {messages_output_path}")

## 8. Limites de cette V1

Cette V1 est volontairement simple. Elle est utile pour poser une base claire, mais elle a plusieurs limites :

- les offres sont entrées manuellement ;
- le scoring dépend de mots-clés exacts ;
- deux formulations proches peuvent ne pas être reconnues si les mots ne sont pas dans la liste ;
- le message de candidature reste basé sur un template ;
- le CV n'est pas encore utilisé automatiquement comme document structuré ;
- l'agent ne cherche pas encore les offres tout seul.

Ces limites sont normales à ce stade. Le but était d'avoir une première chaîne complète et testable.

## 9. Prochaines étapes

Les prochaines étapes utiles pour transformer ce prototype en outil vraiment utilisable sont :

1. utiliser de vraies offres récupérées depuis plusieurs sources ;
2. ajouter un score de similarité sémantique pour dépasser les simples mots-clés ;
3. intégrer un LLM pour générer des messages plus naturels ;
4. structurer le profil candidat à partir du CV ;
5. créer un tableau de suivi des candidatures ;
6. ajouter une interface simple avec Streamlit.

La prochaine étape la plus intéressante pour le projet est probablement la recherche automatique ou semi-automatique d'offres réelles, car c'est ce qui rendra l'outil utile dans une vraie recherche d'emploi.